# Topics Covered:
- [1. Hybrid Retriver: Combine Dense and Sparse Retriever](#1-hybrid-retriver-combine-dense-and-sparse-retriever)
    - [1.1. RAG Pipeline with hybrid retriever](#11-rag-pipeline-with-hybrid-retriever)
- [2. Reranking Hybrid Search Statergies](#2-reranking-hybrid-search-statergies)
    - [2.1. Design the Re Ranker](#21-design-the-re-ranker)
- [3. Maximum Marginal Relavance](#3-maximum-marginal-relavance)

### 1. Hybrid Retriver: Combine Dense and Sparse Retriever

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import EnsembleRetriever, BM25Retriever

from langchain_core.documents import Document

In [3]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers.")
]

In [4]:
# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

In [5]:
### Sparse Retriever(BM25)
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ##top- k documents to retriever

In [ ]:
## step 4 : Combine with Ensemble Retriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)


In [7]:
# Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


#### 1.1. RAG Pipeline with hybrid retriever

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [13]:
openai_key = os.getenv("OPENAI_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

In [9]:
# Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

In [10]:
llm=init_chat_model("openai:gpt-3.5-turbo",temperature=0.2)

In [14]:
### Create stuff Docuemnt Chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

## create Full rAg chain
rag_chain=create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)

In [15]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 You can build an app using LLMs by utilizing LangChain, which helps in developing LLM applications. LangChain can be used to develop agentic AI applications, and it offers various types of retrievers to enhance the functionality of your app. Additionally, you can also consider using Pinecone, a vector database for semantic search, to further optimize the performance of your LLM-based app.

📄 Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application.

Doc 3: Langchain has many types of retrievers.

Doc 4: Pinecone is a vector database for semantic search.


### 2. Reranking Hybrid Search Statergies
Re-ranking is a second-stage filtering process in retrieval systems, especially in RAG pipelines, where we:

1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.

2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

👉 It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [19]:
# load text file
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

In [20]:
# split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(raw_docs)

In [22]:
for index, doc in enumerate(docs[:2]):
    print(f"\n📄 Document Chunk {index + 1}:\n")
    print("=" * 20)
    print("\nMetadata:", doc.metadata)
    print("--"*20)
    print("\nContent:")
    print(doc.page_content)


📄 Document Chunk 1:


Metadata: {'source': 'langchain_sample.txt'}
----------------------------------------

Content:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

📄 Document Chunk 2:


Metadata: {'source': 'langchain_sample.txt'}
----------------------------------------

Content:
LangChain integrates with many third-party services such as OpenAI, Hugging Face, and Cohere. This enables developers to experiment with different models and optimize performance for specific use cases like summarization, question answering, or translation.


In [23]:
## user query
query="How can i use langchain to build an application with memory and tools?"

In [24]:
### FAISS and Huggingface model Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(searc_h_kwargs={"k":8})

In [27]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001ED23F3B390>, search_kwargs={})

In [25]:
retriever.invoke(query)

[Document(id='561f3740-0039-495e-983c-54aedbf8569b', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='f072e7ca-7472-4f2c-bb72-7a69e871bd7a', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='d55db1a6-35bb-416a-afa1-0fe9b17307e5', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

In [26]:
openai_key = os.getenv("OPENAI_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

In [30]:
# using openai embeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")
vectorstore_openai = FAISS.from_documents(docs, embedding_model)
retriever_openai = vectorstore_openai.as_retriever(search_kwargs={"k":8})
retriever_openai

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001ED24171A90>, search_kwargs={'k': 8})

In [31]:
retriever_openai.invoke(query)

[Document(id='d948b75f-1d5b-401a-bf8c-25ada68ed15f', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.\nMemory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.'),
 Document(id='f09eea81-5c3a-46e1-ab7e-bbeb55554efa', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.'),
 Document(id='7dc0ec18-52c4-4174-acec-c6ad82fb39ef', metadata={'source': 'langchain_sample.txt'}, page_content='LangChain integrates with many third-party services such as OpenAI, Hugging F

#### 2.1. Design the Re Ranker

In [32]:
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [33]:
llm=init_chat_model("groq:gemma2-9b-it")

In [35]:
# Prompt Template
prompt = PromptTemplate.from_template("""
You are a helpful assistant. Your task is to rank the following documents from most to least relevant to the user's question.

User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant.

Output format: comma-separated document indices (e.g., 2,1,3,0,...)
""")

In [36]:
chain=prompt| llm | StrOutputParser()

In [37]:
retrieved_docs = retriever_openai.invoke(query)
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)
response=chain.invoke({"question":query,"documents":formatted_docs})
indices = [int(x.strip()) - 1 for x in response.split(",") if x.strip().isdigit()]
reranked_docs = [retrieved_docs[i] for i in indices if 0 <= i < len(retrieved_docs)]

In [38]:
print("\n📊 Final Reranked Results:")
for i, doc in enumerate(reranked_docs, 1):
    print(f"\nRank {i}:\n{doc.page_content}")


📊 Final Reranked Results:

Rank 1:
LangChain is a flexible framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions to work with LLMs more effectively and includes components for prompt management, chains, memory, and agents.

Rank 2:
LangChain supports tool integration including web search, calculators, and APIs, allowing LLMs to interact with external systems and respond more accurately to dynamic queries.
Memory in LangChain enables context retention across multiple steps in a conversation or task, making the application more coherent and stateful.

Rank 3:
FAISS is a popular library used for fast approximate nearest neighbor search in high-dimensional spaces. It supports both flat and compressed indexes, which makes it scalable for large document stores.
Agents in LangChain are chains that use LLMs to decide which tools to use and in what order. This makes them suitable for multi-step tasks like question answering w

### 3. Maximum Marginal Relavance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [39]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [40]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

In [41]:
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# set environment variable for OpenAI API key
if openai_key is not None:
	os.environ["OPENAI_API_KEY"] = openai_key
else:
	raise ValueError("OPENAI_API_KEY environment variable is not set.")

# set environment variable for Groq API key
if groq_key is not None:
    os.environ["GROQ_API_KEY"] = groq_key
else:
    raise ValueError("GROQ_API_KEY environment variable is not set.")

In [43]:
# Step 1: Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [45]:
for i, chunk in enumerate(chunks[:2]):
    print(f"\nChunk {i+1}:")
    print(f"    Source: {chunk.metadata['source']}")
    print(f"    Length: {len(chunk.page_content)} characters")
    print(f"    Content: {chunk.page_content[:100]}...")  # Print first 100 characters
    print("-" * 40)


Chunk 1:
    Source: langchain_rag_dataset.txt
    Length: 265 characters
    Content: LangChain is an open-source framework designed to simplify the development of applications using lar...
----------------------------------------

Chunk 2:
    Source: langchain_rag_dataset.txt
    Length: 243 characters
    Content: The framework supports integration with various vector databases like FAISS and Chroma for semantic ...
----------------------------------------


In [46]:
# Step 2: FAISS Vector Store with HuggingFace Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

In [47]:
### Step 3: Create MMR Retirever
retriever=vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

In [49]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

llm=init_chat_model("groq:gemma2-9b-it")


In [50]:
# Step 5: RAG Pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

In [51]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 LangChain supports agents in several ways:

* **Tool Integration:** Agents can leverage tools like calculators, search APIs, or custom functions based on instructions.
* **API & Database Access:** Agents can interact with external APIs and databases, expanding their capabilities beyond text processing.
* **Tool Selection & Execution:** LangChain enables LLMs to act as agents, deciding which tool to use and the order of execution for a given task.

LangChain supports memory through:

* **Conversational Memory:**  `ConversationBufferMemory` retains previous interactions for coherent multi-turn conversations.
* **Summarization Memory:** `ConversationSummaryMemory` provides a summarized version of past interactions, aiding in context retention. 


Essentially, LangChain empowers LLMs to act as intelligent agents with access to tools and memory, making them more versatile and capable in real-world applications.

